Key steps when using this Colab notebook:

Replace Placeholder URL: In the second code cell, change !git clone https://github.com/git-disl/emotion-classification-dlfa.git {COLAB_PROJECT_ROOT} to your actual project's Git repository URL.

Modify common/config.py: This is CRUCIAL. After cloning/uploading your project and before running the data processing steps:

Open common/config.py using Colab's file browser.
Locate the BASE_DATA_DIR (or equivalent path variable) in your BaseConfig class.
Change its value to point to the location of your meld_data directory on Google Drive. For example: self.BASE_DATA_DIR = Path('/content/drive/MyDrive/your_folder/meld_data/')

Ensure all other paths like raw_dataset_dir, processed_audio_dir, processed_features_dir are correctly derived from this updated BASE_DATA_DIR.

Upload MELD Data: Make sure your raw MELD dataset (CSVs and MP4s) is uploaded to the raw_dataset_dir location on your Google Drive that your modified config.py now points to.

Run Cells Sequentially: Execute the notebook cells one by one.


In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Emotion Classification: Data Processing in Colab"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Setup Environment"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Mount Google Drive (for persistent data storage)\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Clone the repository (replace with your actual repository URL)\n",
    "!git clone https://github.com/YOUR_USERNAME/emotion-classification-dlfa.git\n",
    "%cd emotion-classification-dlfa"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Install FFmpeg (required for audio processing)\n",
    "!sudo apt update && sudo apt install ffmpeg"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create Conda environment from environment.yml (if you have one and prefer Conda)\n",
    "# This can be a bit complex in Colab. Alternatively, install packages via pip.\n",
    "# Example: Install key packages using pip (adjust based on your environment.yml)\n",
    "!pip install torch torchvision torchaudio pandas numpy matplotlib seaborn scikit-learn transformers datasets tqdm wandb ipykernel"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Configure Paths\n",
    "\n",
    "IMPORTANT: Before proceeding, you need to ensure your `common/config.py` uses paths that point to your MELD dataset on Google Drive. \n",
    "For example, if your `meld_data` directory is at `My Drive/Colab_Projects/emotion_classification_dlfa/meld_data`, \n",
    "you might need to update `BASE_DATA_DIR` in `common/config.py` to something like: \n",
    "```python\n",
    "self.BASE_DATA_DIR = Path('/content/drive/MyDrive/Colab_Projects/emotion_classification_dlfa/meld_data/')\n",
    "```\n",
    "Alternatively, you can modify the config object directly in a cell after loading it, before passing it to scripts.\n",
    "Below is an example of how you might load and modify the config dynamically if your scripts/main.py supports it, or you'd edit `common/config.py` after cloning."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Example of how you might load and print config for verification (adapt as needed)\n",
    "# This assumes your config can be instantiated and paths can be set or checked.\n",
    "# from common.config import BaseConfig # Or your specific config if using one directly\n",
    "# cfg = BaseConfig() # Or MLPFusionConfig(), etc.\n",
    "# cfg.BASE_DATA_DIR = '/content/drive/MyDrive/path_to_your_meld_data_on_drive/' # Example override\n",
    "# print(f"Raw data dir: {cfg.raw_dataset_dir}")\n",
    "# print(f"Processed audio dir: {cfg.processed_audio_dir}")\n",
    "# print(f"Processed features dir: {cfg.processed_features_dir}")\n",
    "\n",
    "print("Please ensure your common/config.py is set up for Colab paths or modify the config object dynamically.")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Download MELD Dataset (if not already in Google Drive)\n",
    "\n",
    "If you haven't already, download the MELD dataset and place it in the configured `raw_dataset_dir` on your Google Drive (e.g., `/content/drive/MyDrive/.../meld_data/raw/`).\n",
    "The `scripts/download_meld_dataset.py` can be adapted or run locally to fetch the data first."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Run Data Processing Pipeline"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Step 4.1: Preprocess MELD (MP4s to WAVs)\n",
    "As noted, `main.py --prepare_data` might not fully handle this step due to the placeholder call. \n",
    "It's safer to run `scripts/preprocess_meld.py` directly if your WAV files aren't already generated and in the correct `processed_audio_dir`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Ensure your current directory is the project root if not already\n",
    "# %cd /content/emotion-classification-dlfa \n",
    "\n",
    "# Run the preprocessing script (MP4 to WAV)\n",
    "# This assumes common/config.py is correctly pointing to your raw MP4s and desired output WAV directory on Drive.\n",
    "!python scripts/preprocess_meld.py"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Step 4.2: Build Hugging Face Datasets\n",
    "Once WAV files are generated and in the location specified by `cfg.processed_audio_dir`, \n",
    "you can run the Hugging Face dataset creation step using `main.py --prepare_data`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run the Hugging Face dataset building part of the pipeline\n",
    "# Using --force_reprocess_hf_dataset to ensure it runs even if cached files exist from a previous attempt.\n",
    "# Using limit_dialogues for a potentially faster initial test run - remove for full processing.\n",
    "!python main.py --prepare_data --force_reprocess_hf_dataset --limit_dialogues_train 20 --limit_dialogues_dev 10 --limit_dialogues_test 10"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Verify Output\n",
    "After the scripts complete, check your Google Drive folders:\n",
    "*   `meld_data/processed/audio/` should contain `train/`, `dev/`, `test/` subfolders with WAV files.\n",
    "*   `meld_data/processed/features/hf_datasets/` should contain `train/`, `dev/`, `test/` subfolders with Hugging Face dataset files (e.g., `dataset.arrow`, `state.json`).\n",
    "You can use Colab's file browser or `!ls` commands to check."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Example verification (adjust paths based on your config)\n",
    "# These paths assume your BASE_DATA_DIR in config points to something like '/content/drive/MyDrive/your_project_data_root/meld_data'\n",
    "MEL_DATA_BASE_PATH_ON_DRIVE = '/content/drive/MyDrive/Colab_Projects/emotion-classification-dlfa/meld_data' # Replace with your actual path\n",
    "\n",
    "print("Checking for WAV files...")\n",
    "!ls -l {MEL_DATA_BASE_PATH_ON_DRIVE}/processed/audio/train/ | head -n 5 # Check a few files in train audio\n",
    "!ls -l {MEL_DATA_BASE_PATH_ON_DRIVE}/processed/audio/dev/ | head -n 5   # Check a few files in dev audio\n",
    "\n",
    "print("\nChecking for Hugging Face dataset files...")\n",
    "!ls -l {MEL_DATA_BASE_PATH_ON_DRIVE}/processed/features/hf_datasets/train/ # Check train HF dataset folder\n",
    "!ls -l {MEL_DATA_BASE_PATH_ON_DRIVE}/processed/features/hf_datasets/dev/   # Check dev HF dataset folder"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
